# Lab 12 · Bộ hình chuẩn báo cáo — và assert cho biểu đồ

**Lập trình xử lý dữ liệu (LTXLDL) · 2627-1 · Giờ thực hành tuần 12**

> 💡 File → **Save a copy in Drive** trước khi sửa.

Notebook demo buổi 12 dạy bộ tứ hình. Lab này bắt bạn làm điều bài tập lớn yêu cầu:
**hình sinh tự động từ code, đạt chuẩn xuất bản, lưu vào `figures/`** — và lần đầu tiên,
bạn sẽ thấy biểu đồ cũng *assert được* như mọi kết quả khác.

*Lab ~50 phút; 30 phút cuối là BTL clinic (mục cuối notebook).*

## Cách làm việc trong buổi lab

- Bài tập được chia bước; mỗi bước có ô `TODO` và phần kiểm tra `assert` — chạy qua hết
  `assert` nghĩa là bạn làm đúng.
- Phần khởi động và bài có hướng dẫn: bạn nên **tự gõ, không dùng AI** — các bài kiểm tra
  định kỳ 🔒 ở giờ lý thuyết đo đúng các kỹ năng này.
- Bài tự làm ở cuối được gắn nhãn 🔓: bạn được dùng AI, kèm trách nhiệm khai báo
  theo chính sách AI của môn.
- Bạn kẹt quá 3 phút ở một bước: gọi trợ giảng.

## Mục tiêu

Sau buổi lab, bạn:

1. Dựng 3 hình hoàn chỉnh (title thông điệp, nhãn + đơn vị, nguồn) bằng `fig, ax`.
2. Lưu hình vào `figures/` bằng `savefig` — sản phẩm của pipeline.
3. Kiểm tra hình bằng code: title có chưa, trục bắt đầu từ 0 chưa, đủ số thanh chưa.
4. Sửa một hình "nói dối" thành hình trung thực.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

BASE = "https://data.insideairbnb.com/chile/rm/santiago/2026-06-29/visualisations"
ds = pd.read_csv(f"{BASE}/listings.csv")
rv = pd.read_csv(f"{BASE}/reviews.csv", parse_dates=["date"])
Path("figures").mkdir(exist_ok=True)

plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.25, "font.size": 11})
print("Sẵn sàng — rcParams khai một lần, mọi hình cùng phong cách.")

### Hình 1 · Line — nhịp review của MỘT quận (~15 phút)

Bài tập lớn luôn cần "diễn biến của khu vực X". Ta làm mẫu với Providencia — cần ghép
reviews với listings để biết review nào thuộc quận nào (kỹ năng buổi 5).

In [ ]:
# TODO: lấy tập id các listing thuộc Providencia, lọc rv theo tập đó,
#       rồi đếm theo tháng (resample "ME") và BỎ kỳ cụt cuối
ids_pro = ...
thang_pro = ...

# --- Ô kiểm tra ---
assert len(thang_pro) == 188
assert thang_pro.iloc[-1] == 2810
print(f"{len(thang_pro)} tháng; tháng cuối (06/2026): {thang_pro.iloc[-1]:,} review.")

In [ ]:
# TODO: vẽ line chart hoàn chỉnh và LƯU vào figures/providencia_thang.png
#   - fig, ax = plt.subplots(figsize=(9, 3.5))
#   - ax.plot(...), title mang THÔNG ĐIỆP (không phải "Biểu đồ review"),
#     ylabel có đơn vị, chú thích nguồn bằng fig.text (góc dưới)
#   - fig.savefig("figures/providencia_thang.png", dpi=150, bbox_inches="tight")
fig, ax = plt.subplots(figsize=(9, 3.5))
...

# --- Ô kiểm tra: hình cũng assert được! ---
assert len(ax.lines) >= 1, "Chưa có đường nào trên khung"
assert len(ax.get_title()) >= 15, "Title phải là một thông điệp, không phải nhãn cụt"
assert ax.get_ylabel() != "", "Thiếu nhãn trục y"
assert Path("figures/providencia_thang.png").exists()
print("Hình 1 đạt chuẩn cấu trúc — mở tab Files xem file PNG.")

### Hình 2 · Bar ngang — cơ cấu nguyên căn theo quận (~12 phút)

Chỉ số khác demo: **% nguyên căn** của 8 quận lớn (quận nào "chuyên nghiệp hoá" nhất?).

In [ ]:
# TODO: tính % nguyên căn (room_type == "Entire home/apt") theo quận,
#       chỉ giữ quận >= 300 listing, sắp tăng dần để bar ngang đẹp
ds["nguyen_can"] = ds["room_type"] == "Entire home/apt"
tk = ...          # groupby + agg: n=("id","size"), pct=("nguyen_can","mean")
tk8 = ...         # lọc n >= 300, sort_values("pct")

# TODO: vẽ barh + bar_label (%.0f%%), title thông điệp, LƯU figures/pct_nguyen_can.png
fig, ax = plt.subplots(figsize=(7, 4))
...

# --- Ô kiểm tra ---
assert len(tk8) == 8 and round(tk8["pct"].max() * 100, 1) == 89.9
assert len(ax.patches) == 8, "Phải đúng 8 thanh"
assert ax.get_xlim()[0] == 0, "Trục x của bar phải bắt đầu từ 0"
assert Path("figures/pct_nguyen_can.png").exists()
print("Hình 2 đạt: 8 thanh, trục từ 0, Lo Barnechea dẫn đầu 90%.")

### Hình 3 · Hist — phân phối "hai bướu" (~10 phút)

`availability_365` (số ngày mở lịch trong năm tới) có hình dáng đặc biệt — vẽ mới thấy.

In [ ]:
# TODO: histogram availability_365 với bins=40; title nói đúng điều nhìn thấy;
#       LƯU figures/availability_hist.png
fig, ax = plt.subplots(figsize=(8, 3.5))
...

# --- Ô kiểm tra ---
assert len(ax.patches) == 40
assert Path("figures/availability_hist.png").exists()
so_khoa = (ds["availability_365"] == 0).sum()
so_mo_quanh_nam = (ds["availability_365"] >= 360).sum()
assert so_khoa == 469 and so_mo_quanh_nam == 3789
print(f"Hai bướu: {so_khoa} phòng khoá lịch hoàn toàn — {so_mo_quanh_nam:,} phòng mở gần quanh năm.")

Phân phối "hai bướu" (bimodal) như thế này là lý do **histogram phải được vẽ trước
khi báo cáo trung bình**: mean/median của availability chẳng đại diện cho ai — thị trường
thực ra là hai nhóm hành vi khác hẳn nhau. Một dòng nhận xét như vậy đáng giá hơn ba
con số thống kê.

### Hình 4 · Sửa một hình nói dối (~8 phút)

Ô dưới vẽ đúng số liệu nhưng **trục y bị cắt** làm chênh lệch phồng to. Chạy để xem,
rồi sửa trong ô TODO.

In [ ]:
pct = (ds.groupby("neighbourhood")["nguyen_can"].mean() * 100).loc[
    ["Providencia", "Vitacura", "La Florida", "Las Condes"]].sort_values()

fig, ax = plt.subplots(figsize=(6, 3))
ax.bar(pct.index, pct.values, color="#E62727")
ax.set_ylim(70, 85)          # <-- thủ phạm
ax.set_title("Vitacura 'thấp hơn hẳn' Las Condes?!")
plt.show()

In [ ]:
# TODO: vẽ lại ĐÚNG — bar bắt đầu từ 0, title trung thực với chênh lệch thật
fig, ax = plt.subplots(figsize=(6, 3))
...

# --- Ô kiểm tra ---
assert ax.get_ylim()[0] == 0, "Bar chart: trục y bắt đầu từ 0 — không có ngoại lệ"
assert len(ax.patches) == 4
print("Bản sửa đạt: cùng số liệu, ấn tượng trung thực — 4 quận thật ra khá sát nhau.")

## Bài tự làm 🔓

**Bộ hình cho thành phố của nhóm.** Lặp lại Hình 1–3 cho **thành phố bài tập lớn của
nhóm bạn** (đổi URL snapshot). Mỗi hình phải qua đủ 3 assert cấu trúc (title ≥15 ký tự,
nhãn trục, file trong `figures/`). Đây chính là một phần deliverable "≥6 hình" của đề —
làm ở đây là làm luôn cho bài tập lớn.

In [ ]:
# Viết bài tự làm của bạn ở đây

---

## 🧭 BTL clinic tuần 12 (~30 phút — theo nhóm)

Trọng tâm: **dàn hình cho báo cáo.**

1. ☐ Thư mục `figures/` của repo sinh **tự động** khi chạy pipeline (không dán ảnh chụp tay).
2. ☐ Danh sách ≥6 hình dự kiến của báo cáo: mỗi hình một dòng "thông điệp" viết TRƯỚC
   khi vẽ (hình không có thông điệp = chưa cần vẽ).
3. ☐ Ít nhất 2 hình đã chạy được với dữ liệu thành phố của nhóm (line thời gian + một
   hình so sánh nhóm).
4. ☐ Mọi hình dùng chung một rcParams/style — phong cách nhất quán là tiêu chí chấm.
5. ☐ Hình so sánh 2 snapshot (nếu có) dùng chung thang đo (`sharey`/cùng ylim).

> Nhóm xong sớm: chạy 4 assert cấu trúc của lab trên chính hình của nhóm.

## Tóm tắt buổi lab

| Bạn đã làm | Dùng cho |
|---|---|
| 3 hình chuẩn xuất bản, savefig vào figures/ | deliverable "≥6 hình" của đề |
| Assert cho biểu đồ (title/trục/số thanh/file) | tự kiểm hình như kiểm số |
| Phát hiện bimodal bằng hist | tránh báo cáo mean vô nghĩa |
| Sửa trục y bị cắt | trung thực thị giác (buổi 12–13) |

Buổi lý thuyết tới: **seaborn, bản đồ, và phê bình biểu đồ AI**.